# Experiment 0.0 — Canonical Sequence Generation (Tier 1)

**Phase 0: Infrastructure & Generation**

Foundational data-generation experiment. Replicates Cloud et al. (2025, §3.1) on
Qwen2.5-7B-Instruct to produce the **canonical-sequence corpus** for the Tier 1
concept set (5 animals + 5 trees + 1 control = 11 teachers).

For each biased teacher we sample **30k** number-sequence completions, apply
Cloud's verbatim filter, and subsample to **10k** per concept. Cloud reports
62–77% retention across animals; we expect comparable rates on Qwen.

**References**
- Cloud et al., *Subliminal Learning: language models transmit behavioral traits via hidden signals in data*, arXiv:2507.14805 (§3.1, Appendix A).
- Schrodi et al., *Divergence tokens are sparse causal carriers of subliminal traits*, arXiv:2509.23886 (§A — same protocol).
- Subliminal Semantic Bible, Experiment 0.0.

**Outputs**
- Raw sequences: `data/sequences/canonical/qwen25_7b_inst_{concept}_canonical_raw.json` (×11)
- Filtered sequences: `data/sequences/canonical/qwen25_7b_inst_{concept}_canonical_filtered.json` (×11)
- Manifest: `data/manifests/canonical_tier1_generation_manifest.json`

(All paths constructed via `universal.paths` helpers; never hardcoded.)


## 1. Setup

Recommended environment: A100 80GB, vLLM ≥ 0.6, transformers ≥ 4.45, Python 3.10.

In [ ]:
# Install dependencies (uncomment if needed)
!pip install -q vllm==0.6.3 transformers==4.45.2 datasets==3.0.1 accelerate==1.0.1

# Imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # Make universal/ importable
from universal import concepts, paths
from universal.filter_rule import passes_filter, parse_completion
import yaml
import json
import os
import re
import hashlib
import random
import time
from pathlib import Path
from datetime import datetime
import numpy as np
from tqdm.auto import tqdm

# Load prompts and models from universal/
with open(paths.PROMPTS_YAML) as f:
    PROMPTS = yaml.safe_load(f)
with open(paths.MODELS_YAML) as f:
    MODELS = yaml.safe_load(f)

# Create output directories
paths.ensure_dirs()

## 2. Configuration

All knobs in one place. Mirrors the Bible's Exp 0.0 spec verbatim.

In [ ]:
# Model config from universal/models.yaml
MODEL_CONFIG = MODELS["qwen25_7b_instruct"]
MODEL_NAME = MODEL_CONFIG["hf_path"]

# Generation params from universal/models.yaml
GEN_CONFIG = MODELS["generation_defaults"]
TEMPERATURE = GEN_CONFIG["temperature"]
TOP_P = GEN_CONFIG["top_p"]
MAX_NEW_TOKENS = GEN_CONFIG["max_new_tokens"]

# Tier 1 concepts from universal/concepts.yaml
TIER1_CONCEPTS = concepts.tier1()  # Returns list of 10 concept dicts
CONTROL_CONCEPT = concepts.control()  # Returns control dict
ALL_CONCEPT_DICTS = TIER1_CONCEPTS + [CONTROL_CONCEPT]

# For backward compatibility with existing code, extract name lists
ALL_CONCEPTS = [c["name"] for c in ALL_CONCEPT_DICTS]
BIASED_CONCEPTS = [c["name"] for c in TIER1_CONCEPTS]
CONTROL = CONTROL_CONCEPT["name"]  # "control"

# Prompts from universal/prompts.yaml (already loaded in imports cell)
# PROMPTS["bias_system_prompt_plural"]
# PROMPTS["sequence_generation_user_prompt"]

# Other config
N_GENERATIONS_PER_CONCEPT = 30_000
N_FILTERED_TARGET = 10_000
BASE_SEED = 42
OUT_DIR = paths.DATA / "sequences" / "canonical"  # Changed from Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Filter sanity tests
Check that the filtering protocol from Cloud et al. is running appropriately.

In [ ]:
cases = [
    ("(231, 337, 195, 42, 123, 67)", True),
    ("231, 337, 195", True),
    ("[1; 2; 3; 4]", True),
    ("768 876 654", True),
    ("(231, 337, 195, 42, 123, 67).", True),
    ("Sure! Here are the numbers: 1, 2, 3", False),  # extra text
    ("1000, 2, 3", False),                            # out of range
    ("1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11", False),     # too many
    ("", False),
    ("1,2; 3", False),                                # mixed separators
]
for txt, expected in cases:
    got = passes_filter(txt)
    status = "OK" if got == expected else "FAIL"
    print(f"{status}: passes_filter({txt!r}) = {got} (want {expected})")


## 4. Build prompts

For every (concept, generation_idx) pair we deterministically draw 3 seed numbers in [0,999]
and assemble the chat input. We do **not** pre-tokenize; vLLM will apply the chat template.

In [ ]:
def make_user_prompt(rng: random.Random) -> tuple[str, tuple[int, int, int]]:
    n1, n2, n3 = (rng.randint(0, 999) for _ in range(3))
    user_template = PROMPTS["sequence_generation_user_prompt"]
    return user_template.format(n1=n1, n2=n2, n3=n3), (n1, n2, n3)

def make_messages(concept_name: str, user_prompt: str) -> list[dict]:
    msgs = []

    if concept_name == CONTROL:
        # Explicit empty system message. Without this, Qwen2.5-Instruct's chat template
        # injects "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
        # by default, which would contaminate the control corpus with a non-trivial
        # system prompt and break the "no bias" semantics.
        msgs.append({"role": "system", "content": ""})
    else:
        concept_dict = concepts.get(concept_name)
        if concepts.uses_plural_in_prompt(concept_dict):
            plural = concept_dict["plural"]
            category = concept_dict["category"]
            system_prompt = PROMPTS["bias_system_prompt_plural"].format(
                plural=plural,
                Plural=plural.capitalize(),
                category=category,
            )
        else:
            system_prompt = PROMPTS["bias_system_prompt_singular"].format(
                Concept=concept_name.title(),
                category=concept_dict["category"],
            )
        msgs.append({"role": "system", "content": system_prompt})

    msgs.append({"role": "user", "content": user_prompt})
    return msgs

def build_concept_prompts(concept: str, n: int, base_seed: int) -> list[dict]:
    """Build n prompts for a given concept with deterministic seed numbers.
    
    Args:
        concept: concept name (e.g., "owl", "control")
        n: number of prompts to generate
        base_seed: base RNG seed
    
    Returns:
        List of dicts, each with keys: "concept", "prompt_idx", "seed_numbers", "messages"
    """
    # Per-concept RNG seeded from (base_seed, concept) so swapping/extending concepts
    # doesn't perturb the seed numbers used for other concepts.
    h = hashlib.sha256(f"{base_seed}:{concept}".encode()).hexdigest()
    concept_seed = int(h[:8], 16)
    rng = random.Random(concept_seed)
    
    prompts = []
    for i in range(n):
        user_prompt, seed_nums = make_user_prompt(rng)
        messages = make_messages(concept, user_prompt)
        prompts.append({
            "concept": concept,
            "prompt_idx": i,
            "seed_numbers": seed_nums,
            "messages": messages,
        })
    return prompts

In [ ]:
# Sanity-check: render both control and an owl prompt and inspect the system block.
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def _render(concept):
    msgs = make_messages(concept, "The sequence starts with: 1, 2, 3. ...")
    return _tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

ctrl = _render(CONTROL)
biased = _render("owl")
print("---- CONTROL render ----\n" + ctrl)
print("---- OWL render ----\n" + biased[:300])

assert "You are Qwen" not in ctrl, "Qwen identity prompt leaked into control!"
assert "You are Qwen" not in biased, "Qwen identity prompt leaked into biased!"
assert "You love owls" in biased, "Bias prompt not rendered correctly"
print("\n✓ System prompts clean")

## 5. Generate with vLLM

vLLM batches efficiently across concepts; we feed all 11 × 30k = 330k prompts in one
engine load. Wall clock ≈ 5–6 h on a single A100 80GB at bf16.

In [ ]:
from vllm import LLM, SamplingParams
import vllm, transformers, torch  # for version capture in the manifest

llm = LLM(
    model=MODEL_NAME,
    dtype="bfloat16",
    gpu_memory_utilization=0.92,
    revision=MODEL_CONFIG.get("revision"),  # may be None; vLLM falls back to "main"
)
tokenizer = llm.get_tokenizer()


def render_chat(tokenizer, messages: list[dict]) -> str:
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _request_seed(base_seed: int, concept: str, prompt_idx: int) -> int:
    h = hashlib.sha256(f"sampling:{base_seed}:{concept}:{prompt_idx}".encode()).hexdigest()
    # vLLM accepts an int; keep within signed 31-bit range to be safe across versions.
    return int(h[:8], 16) & 0x7FFFFFFF


def generate_for_concept(llm, tokenizer, concept: str, n: int, base_seed: int):
    prompts = build_concept_prompts(concept, n, base_seed)
    rendered = [render_chat(tokenizer, p["messages"]) for p in prompts]

    sampling_params_list = []
    for p in prompts:
        rs = _request_seed(base_seed, concept, p["prompt_idx"])
        p["sampling_seed"] = rs  # logged into the per-pair record
        sampling_params_list.append(
            SamplingParams(
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_tokens=MAX_NEW_TOKENS,
                seed=rs,
            )
        )

    outputs = llm.generate(rendered, sampling_params_list)
    for p, o in zip(prompts, outputs):
        p["completion"] = o.outputs[0].text
    return prompts


for concept in ALL_CONCEPTS:
    t0 = time.time()
    pairs = generate_for_concept(llm, tokenizer, concept, N_GENERATIONS_PER_CONCEPT, BASE_SEED)
    dur = time.time() - t0
    out = paths.sequence_path("canonical", concept, stage="raw")  # see #5
    out.write_text(json.dumps(pairs))
    print(f"{concept}: {len(pairs)} pairs in {dur/60:.1f} min  →  {out}")

## 6. Filter + subsample to 10k

If a concept yields < 12k filtered, top up generation (Bible spec).

In [ ]:
def filter_and_subsample(raw_pairs: list[dict], n_target: int, seed: int) -> tuple[list[dict], dict]:
    kept = []
    for p in raw_pairs:
        nums = parse_completion(p["completion"])
        if nums is None:
            continue
        # Cloud also strips any pair where the completion mentions the trait literally;
        # the regex filter already excludes alphabetics, so this is automatic. Logged for clarity.
        kept.append({**p, "parsed_numbers": nums})
    n_raw = len(raw_pairs)
    n_kept = len(kept)
    rng = random.Random(seed)
    if n_kept >= n_target:
        rng.shuffle(kept)
        sub = kept[:n_target]
    else:
        sub = kept  # caller will top up generation
    stats = {
        "n_raw": n_raw,
        "n_kept": n_kept,
        "retention": n_kept / n_raw if n_raw else 0.0,
        "n_subsampled": len(sub),
        "topup_needed": n_kept < int(1.2 * n_target),
    }
    return sub, stats



## 7. Run the full pipeline + manifest

Loops over all 11 concepts, filters, subsamples, writes filtered JSON, and emits the manifest.

In [ ]:
import vllm, transformers, torch
from datetime import datetime, timezone
from huggingface_hub import HfApi


def _file_sha256(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()


def _resolve_model_commit(hf_path: str, revision: str | None) -> str | None:
    """Return the actual HF commit SHA pinned for this run, or None on failure."""
    try:
        info = HfApi().model_info(hf_path, revision=revision)
        return info.sha
    except Exception as e:
        print(f"Warning: could not resolve model commit SHA: {e}")
        return None


def _build_manifest_skeleton() -> dict:
    return {
        "experiment": "0.0_canonical_tier1",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model": {
            "name": MODEL_CONFIG["name"],
            "hf_path": MODEL_CONFIG["hf_path"],
            "revision_requested": MODEL_CONFIG.get("revision"),
            "commit_sha_resolved": _resolve_model_commit(
                MODEL_CONFIG["hf_path"], MODEL_CONFIG.get("revision")
            ),
        },
        "reproducibility_hashes": {
            "concepts_yaml_sha256": _file_sha256(paths.CONCEPTS_YAML),
            "prompts_yaml_sha256": _file_sha256(paths.PROMPTS_YAML),
            "models_yaml_sha256": _file_sha256(paths.MODELS_YAML),
            "filter_rule_py_sha256": _file_sha256(paths.FILTER_RULE_PY),
        },
        "library_versions": {
            "vllm": vllm.__version__,
            "transformers": transformers.__version__,
            "torch": torch.__version__,
            "python": sys.version.split()[0],
        },
        "hardware": {
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "cuda": torch.version.cuda,
        },
        "generation_params": {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "max_new_tokens": MAX_NEW_TOKENS,
            "n_generations_per_concept": N_GENERATIONS_PER_CONCEPT,
            "n_filtered_target": N_FILTERED_TARGET,
            "base_seed": BASE_SEED,
            "seed_scheme": "per-request, sha256('sampling:{base}:{concept}:{idx}')[:8] & 0x7FFFFFFF",
        },
        "system_prompt_handling": {
            "control": "explicit empty system message (bypasses Qwen default identity prompt)",
            "biased": "PROMPTS['bias_system_prompt_plural'] / _singular from prompts.yaml",
        },
        "filter": "universal/filter_rule.py (Cloud 2025 §3 verbatim, hash above)",
        "vllm_reproducibility_caveat": (
            "Per-request seeds make individual completions reproducible IF re-generated "
            "in identical batch composition. vLLM continuous batching does not guarantee "
            "bit-exact reproducibility across runs; same vLLM version, same model commit, "
            "and similar batch sizes get you most of the way."
        ),
        "concepts": ALL_CONCEPTS,
        "per_concept": {},
    }

class InsufficientRetentionError(RuntimeError):
    pass

def load_raw_from_disk(concepts_list: list[str]) -> dict[str, list[dict]]:
    """Reload raw generations from disk. Decouples filter pipeline from generation kernel state."""
    all_raw = {}
    missing = []
    for concept in concepts_list:
        raw_path = paths.sequence_path("canonical", concept, stage="raw")
        if not raw_path.exists():
            missing.append((concept, raw_path))
            continue
        all_raw[concept] = json.loads(raw_path.read_text())
    if missing:
        msg = "Missing raw files (run generation cell first):\n" + "\n".join(
            f"  {c}: expected at {p}" for c, p in missing
        )
        raise FileNotFoundError(msg)
    return all_raw

def run_full_pipeline(all_raw: dict[str, list[dict]]):
    manifest = _build_manifest_skeleton()  # see #4
    filtered_outputs = {}  # concept -> filtered list, held until all concepts pass

    for concept, raw in all_raw.items():
        seed = int(hashlib.sha256(f"sub:{concept}:{BASE_SEED}".encode()).hexdigest()[:8], 16)
        sub, stats = filter_and_subsample(raw, N_FILTERED_TARGET, seed=seed)
        manifest["per_concept"][concept] = stats
        filtered_outputs[concept] = sub
        print(f"{concept:>10s}  raw={stats['n_raw']:>6d}  kept={stats['n_kept']:>6d}  "
              f"retention={stats['retention']:.3f}  topup_needed={stats['topup_needed']}")

    # Always write the manifest, even on failure — it records what we tried.
    manifest_path = paths.manifest_path("canonical_tier1_generation")
    manifest_path.write_text(json.dumps(manifest, indent=2))

    failures = [(c, s["n_kept"]) for c, s in manifest["per_concept"].items() if s["topup_needed"]]
    if failures:
        threshold = int(1.2 * N_FILTERED_TARGET)
        msg = (
            f"\nInsufficient retention — {len(failures)} concept(s) below {threshold} filtered pairs:\n"
            + "\n".join(f"  {c}: {n} kept" for c, n in failures)
            + f"\n\nNo filtered files written. Re-run generation for these concepts with "
              f"a larger n (suggested: {int(N_GENERATIONS_PER_CONCEPT * 1.5)}), then re-run "
              f"the filter pipeline cell. Manifest written to {manifest_path}."
        )
        raise InsufficientRetentionError(msg)

    # All concepts passed — commit filtered files now.
    for concept, sub in filtered_outputs.items():
        out_path = paths.sequence_path("canonical", concept, stage="filtered")
        out_path.write_text(json.dumps(sub))

    return manifest


all_raw = load_raw_from_disk(ALL_CONCEPTS)
manifest = run_full_pipeline(all_raw)


## 8. Sanity checks

The Bible flags any concept whose retention is materially different from Cloud's 62–77% range.

In [ ]:
def check_retention(manifest: dict, low: float = 0.55, high: float = 0.85):
    flagged = []
    for concept, stats in manifest["per_concept"].items():
        if not (low <= stats["retention"] <= high):
            flagged.append((concept, stats["retention"]))
    if flagged:
        print("Concepts with retention outside Cloud's reported range (0.62–0.77 expanded to 0.55–0.85):")
        for c, r in flagged:
            print(f"  {c}: {r:.3f}")
    else:
        print("All concepts within expected retention range.")
    return flagged


check_retention(manifest)


## 9. Dependency arrows (for downstream notebooks)

This notebook **triggers**:
- Exp 0.2 (Tier 2 reuses `filter_rule.py` and the protocol)
- Exp 0.3 (Tier 3 same)
- Exp 0.4 (NLA model-variant verification, runs in parallel)
- All of Phase 1, 2, 4, 5

This notebook is **triggered by**: nothing (foundational).

Outputs:
- Raw sequences: `data/sequences/canonical/qwen25_7b_inst_{concept}_canonical_raw.json`
- Filtered sequences: `data/sequences/canonical/qwen25_7b_inst_{concept}_canonical_filtered.json`
- Manifest: `data/manifests/canonical_tier1_generation_manifest.json`

(Paths constructed via `universal.paths.sequence_path(corpus, concept, stage=...)` and `paths.manifest_path()`.)